# Generate maps

This notebook turns a trained model into an interactive prediction map: the predicted suitability of a species across space. You can overlay the sPlotOpen evaluation plots and use a threshold slider to see where the model gets things right and where it doesn't.

The maps are built live here, using a provided checkpoint (the Optimized DeepSDM, `optimized_deepsdm_s2`, see `configs/paper_runs.yaml`) and the full geographic data. Swap `SPECIES` or `CKPT` for your own.

> ⚠️ **Prerequisite 1 — per-species range polygons.** This notebook draws each species' POWO native range, which ships separately from the core data (it's large — only the map notebooks need it). Download `native_ranges.tar.gz` from the [Zenodo data record](https://doi.org/10.5281/zenodo.21297133) and extract it into your `data_dir`:
>
> ```bash
> cd <your data_dir> && tar xzf native_ranges.tar.gz   # creates native_ranges/
> ```

> ⚠️ **Prerequisite 2 — a trained checkpoint.** Maps are predicted live, so this notebook needs one run from the model record (a different Zenodo record from the data). Download `checkpoints.tar` and extract just the one run into your `log_dir` — naming the run folder avoids unpacking all 25:
>
> ```bash
> # $LOG_DIR is the log_dir from configs/local_default.yaml
> tar xf checkpoints.tar -C "$LOG_DIR" optimized_deepsdm_s2     
> tar xf configs.tar     -C "$LOG_DIR" optimized_deepsdm_s2      
> ```
>
> That gives `$LOG_DIR/optimized_deepsdm_s2/{checkpoints/best.ckpt, config.yaml}`. Both are required — the checkpoint holds weights only, and `config.yaml` is what tells the loader which network to put them into. Ending the command with the run-folder name extracts only that folder, instead of unpacking all 25 runs. Write the name literally — a glob such as `'optimized_deepsdm_s2/*'` is rejected by GNU tar on Linux.
>
> The confusion overlay near the end additionally needs that run's predictions and the model record's ground truth (which carries plot coordinates; the data record's copy does not):
>
> ```bash
> tar xf predictions.tar -C "$LOG_DIR" optimized_deepsdm_s2     
> cp eval_test_targets.h5 "$LOG_DIR"/                            
> ```
>
> Both are optional: without them the prediction map still renders and only the overlay is skipped.
>
> See [`docs/MODELS.md`](../docs/MODELS.md#the-released-runs-model-record) for the full record layout.

In [ ]:
# Prerequisite check — per-species range polygons (see the note above).
import os, rootutils, yaml
rootutils.setup_root(os.path.abspath(""), indicator=".project-root", pythonpath=True, cwd=True)
_dd = yaml.safe_load(open("configs/local_default.yaml"))["data_dir"]
_nr = os.path.join(_dd, "native_ranges")
assert os.path.isdir(_nr) and os.listdir(_nr), (
    f"Native range polygons not found in {_nr}. Download native_ranges.tar.gz from the "
    f"Zenodo record and extract it into your data_dir:  cd {_dd} && tar xzf native_ranges.tar.gz"
)

In [6]:
import os
import glob
import yaml
import h5py
import rootutils
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"
ROOT = rootutils.setup_root(os.path.abspath(""), indicator=".project-root", pythonpath=True, cwd=True)
from src.visualizations.maps import (get_species_id, load_species_data, load_world_map,
    load_bg_coords, compute_crop_bounds, build_prediction_grid, predict_with_model, make_interactive_map)
from src.visualizations.maps.render import make_prediction_map_fig, set_continuous_colorscale

cfg = yaml.safe_load(open("configs/local_default.yaml"))
data_dir, log_dir = cfg["data_dir"], cfg["log_dir"]
_ckpts = (sorted(glob.glob(f"{log_dir}/**/optimized_deepsdm_s2/checkpoints/*.ckpt", recursive=True))
          or sorted(glob.glob(f"{log_dir}/**/optimized_deepsdm_*/checkpoints/*.ckpt", recursive=True)))
_HOWTO = (
    "From the Zenodo model record (10.5281/zenodo.21335338), extract just this run:\n"
    f'    tar xf checkpoints.tar -C "{log_dir}" optimized_deepsdm_s2\n'
    f'    tar xf configs.tar     -C "{log_dir}" optimized_deepsdm_s2\n'
    "(If you already extracted them, check that log_dir in configs/local_default.yaml points there.)"
)
assert _ckpts, f"No optimized_deepsdm checkpoint under log_dir={log_dir!r}.\n{_HOWTO}"
CKPT = _ckpts[-1]
RUN  = os.path.dirname(os.path.dirname(CKPT))
# The checkpoint stores weights only — config.yaml is what rebuilds the network around them.
assert os.path.exists(f"{RUN}/config.yaml"), f"No config.yaml in {RUN}.\n{_HOWTO}"
print("using checkpoint:", CKPT)

SPECIES = "Quercus longinux"                 # change species here, but note: large ranges and many occurrences will take longer to predict and plot
set_continuous_colorscale("magma_cb")
sid = get_species_id(SPECIES, data_dir)
_, _, range_gdf = load_species_data(SPECIES, data_dir)
crop = compute_crop_bounds(range_gdf)
world, bg = load_world_map(data_dir), load_bg_coords(data_dir)

# predict suitability on a 0.05° grid over the species' range (~a few seconds). Change resolution according to range size
lons, lats, gc, gp, land_idx = build_prediction_grid(crop, world, bg, data_dir, res=0.05)
probs = predict_with_model(CKPT, gc, gp, species_col=sid)

# The sPlotOpen evaluation plots for this species, for the confusion overlay further down.
# This needs the run's test_predictions.h5 (a separate ~1.9 GB file in the model record) —
# the prediction map itself only needs the checkpoint, so carry on without it if it is absent.
PRED_H5 = f"{RUN}/test_predictions.h5"

# Ground truth: prefer the MODEL record's eval_test_targets.h5 (copied into log_dir) —
# it carries plot `locations`, which the data record's targets-only copy does not, and
# test_predictions.h5 holds `probabilities` alone.
GT_H5 = (f"{log_dir}/eval_test_targets.h5" if os.path.exists(f"{log_dir}/eval_test_targets.h5")
         else f"{data_dir}/targets/eval_test_targets.h5")
with h5py.File(GT_H5) as f:
    T = f["targets"][:, sid]
    L = f["locations"][:] if "locations" in f else None

HAVE_PREDS = os.path.exists(PRED_H5) and L is not None
if HAVE_PREDS:
    with h5py.File(PRED_H5) as f:
        P = f["probabilities"][:, sid]
    in_range = T != -1
    print(f"{len(land_idx):,} land cells predicted · {int(in_range.sum()):,} sPlotOpen plots in range")
else:
    missing = ("this run's test_predictions.h5" if not os.path.exists(PRED_H5)
               else f"plot coordinates ({GT_H5} has no `locations`)")
    print(f"{len(land_idx):,} land cells predicted.\n"
          f"The confusion overlay below will be skipped — missing {missing}.\n"
          f'    tar xf predictions.tar -C "{log_dir}" optimized_deepsdm_s2\n'
          f'    cp eval_test_targets.h5 "{log_dir}"/')

  BG points in crop: 1,435
  Building KD-tree on crop bg coords …
  KD-tree done.
  Prediction grid: 75×55, land cells: 1,465
  Loading bg_chelsa.h5 …
  Loading bg_soilgrids250.h5 …
  Loading bg_topography.h5 …
  Loading bg_human_footprint.h5 …


/opt/conda/envs/sdm/lib/python3.10/site-packages/torch/nn/modules/lazy.py:181: UserWarning:

Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.



1,465 land cells predicted · 550 sPlotOpen plots in range


## The prediction map

`make_prediction_map_fig` renders the suitability surface as an interactive plotly map. Darker means higher predicted suitability, and the POWO native range is outlined.

In [7]:
fig = make_prediction_map_fig(lons, lats, land_idx, probs, crop, range_gdf, world, title=SPECIES)
fig.update_layout(width=540, height=500, margin=dict(t=40, l=10, r=10, b=10))
fig.show(config={"scrollZoom": True})

## Threshold slider and confusion overlay

`make_interactive_map` adds the sPlotOpen plots as a confusion overlay with a threshold slider. It opens at a 0.5 cut-off, so the map shows the model's presence/absence decision.

You can toggle any class on or off in the legend (true negatives start off). Drag the slider to change the threshold, all the way down to *continuous* for the raw probability surface, and watch precision and recall trade off.

> Resolution note: the coloured surface uses a coarser prediction grid than the point-level scores that classify the plots. Near the threshold, a plot can end up on a cell binarised to the *other* class, for example a true negative sitting on a "present" cell, because the grid cell's centre and the plot's exact location fell on opposite sides of the cut-off. That's just a rendering artifact of the resolution mismatch; each plot's own class, and the counts, are correct.

In [ ]:
if HAVE_PREDS:
    fig = make_interactive_map(lons, lats, land_idx, probs,
                               L[in_range], T[in_range], P[in_range],
                               crop, range_gdf, world, title=SPECIES)
    fig.update_layout(width=560, height=560)
    fig.show(config={"scrollZoom": True})
else:
    print("Skipped: needs this run's test_predictions.h5 (see the note above).")

## Export from the command line

For any species and any checkpoint, `scripts/generate_maps.py` writes a self-contained interactive HTML map, plus a static PNG with `--png`. This is handy for large-range species or batch jobs.

```bash
python scripts/generate_maps.py \
    --model   logs/optimized_deepsdm_s2/checkpoints/best.ckpt \
    --species "Fagus sylvatica" \
    --colorscheme magma_cb \
    --out     maps/fagus.html --png
```

Here's the result for *Fagus sylvatica* (European beech), a full continental map.

<img src="assets/map_fagus.png" width="560" alt="Predicted suitability of Fagus sylvatica across Europe">

